In [62]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import string, math, random
import numpy as np
import torch
from scipy.sparse import csr_matrix, diags
from collections import defaultdict, Counter
from sklearn.metrics import f1_score
from sklearn.feature_selection import chi2
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('Imports done.')


Device : cuda
GPU    : NVIDIA GeForce RTX 5080 Laptop GPU
Imports done.


In [63]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
def load_train(path):
    labels, texts = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split(' ', 1)
                labels.append(int(parts[0]))
                texts.append(parts[1] if len(parts) > 1 else '')
    return labels, texts

def load_test(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(line)
    return texts

train_labels, train_texts = load_train('train.dat')
test_texts = load_test('test.dat')
print(f'Training samples : {len(train_labels)}')
print(f'Test samples     : {len(test_texts)}')
print(f'Classes          : {sorted(set(train_labels))}')

Training samples : 102080
Test samples     : 25520
Classes          : [1, 2, 3, 4]


In [64]:
# ── Cell 3: Pre-processing (unigrams + bigrams + trigrams) ────────────────────
STOP_WORDS  = set(stopwords.words('english'))
stemmer     = PorterStemmer()
PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def preprocess(text):
    text     = text.lower().translate(PUNCT_TABLE)
    unigrams = [stemmer.stem(t) for t in text.split()
                if t not in STOP_WORDS and len(t) > 1]
    bigrams  = [f'{unigrams[i]}_{unigrams[i+1]}'
                for i in range(len(unigrams) - 1)]
    trigrams = [f'{unigrams[i]}_{unigrams[i+1]}_{unigrams[i+2]}'
                for i in range(len(unigrams) - 2)]
    return unigrams + bigrams + trigrams

print(preprocess('oil prices rise in the stock market today'))

['oil', 'price', 'rise', 'stock', 'market', 'today', 'oil_price', 'price_rise', 'rise_stock', 'stock_market', 'market_today', 'oil_price_rise', 'price_rise_stock', 'rise_stock_market', 'stock_market_today']


In [65]:
# ── Cell 4: Apply Pre-processing ─────────────────────────────────────────────
print('Preprocessing training data...')
train_tokens = [preprocess(t) for t in train_texts]
print('Preprocessing test data...')
test_tokens  = [preprocess(t) for t in test_texts]
print(f'Done. Sample: {train_tokens[0][:8]}')

Preprocessing training data...
Preprocessing test data...
Done. Sample: ['town', 'celebr', 'hope', 'amir', 'strike', 'gold', 'amir', 'khan']


In [66]:
# ── Cell 5: Helper Functions ──────────────────────────────────────────────────

def renormalize(mat):
    """L2-normalize each row of a sparse matrix."""
    norms = np.sqrt(mat.multiply(mat).sum(axis=1)).A1
    norms[norms == 0] = 1
    return diags(1.0 / norms) @ mat

def build_vocab_idf(token_lists, min_df=2, max_vocab=150000, mode='bm25'):
    """Build vocabulary and IDF array from token lists."""
    N = len(token_lists)
    doc_freq = defaultdict(int)
    for toks in token_lists:
        for t in set(toks):
            doc_freq[t] += 1
    terms = sorted([(t, df) for t, df in doc_freq.items() if df >= min_df],
                   key=lambda x: -x[1])[:max_vocab]
    vocab = {t: i for i, (t, _) in enumerate(terms)}
    idf   = np.zeros(len(vocab), dtype=np.float32)
    for t, idx in vocab.items():
        df = doc_freq[t]
        if mode == 'bm25':
            idf[idx] = math.log((N - df + 0.5) / (df + 0.5) + 1)
        else:
            idf[idx] = math.log((N + 1) / (df + 1)) + 1
    return vocab, idf, doc_freq

def build_bm25_matrix(token_lists, vocab, idf, k1, b, avgdl):
    """Build BM25-weighted L2-normalized sparse matrix."""
    rows, cols, vals = [], [], []
    for doc_idx, tokens in enumerate(token_lists):
        if not tokens: continue
        tf_counts = Counter(tokens)
        doc_len   = len(tokens)
        for term, tf in tf_counts.items():
            if term in vocab:
                col   = vocab[term]
                score = idf[col] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avgdl))
                rows.append(doc_idx); cols.append(col); vals.append(score)
    mat = csr_matrix((vals, (rows, cols)), shape=(len(token_lists), len(vocab)), dtype=np.float32)
    return renormalize(mat)

def build_tfidf_matrix(token_lists, vocab, idf):
    """Build sublinear TF-IDF L2-normalized sparse matrix."""
    rows, cols, vals = [], [], []
    for doc_idx, tokens in enumerate(token_lists):
        if not tokens: continue
        tf_counts = Counter(tokens)
        total     = len(tokens)
        for term, count in tf_counts.items():
            if term in vocab:
                col = vocab[term]
                rows.append(doc_idx); cols.append(col)
                vals.append((1 + math.log(count)) / total * idf[col])
    mat = csr_matrix((vals, (rows, cols)), shape=(len(token_lists), len(vocab)), dtype=np.float32)
    return renormalize(mat)

def apply_chi2(train_mat, test_mat, labels, k):
    """Select top-k features by chi-squared score, re-normalize."""
    scores, _ = chi2(train_mat, labels)
    top = np.argsort(scores)[-k:]; top.sort()
    return renormalize(train_mat[:, top].tocsr()), renormalize(test_mat[:, top].tocsr())

def scipy_sparse_to_torch(mat, device):
    mat = mat.tocsr().astype(np.float32)
    mat.sort_indices(); mat.sum_duplicates()
    crow   = torch.tensor(mat.indptr,  dtype=torch.int64)
    col    = torch.tensor(mat.indices, dtype=torch.int64)
    values = torch.tensor(mat.data,    dtype=torch.float32)
    return torch.sparse_csr_tensor(crow, col, values, size=mat.shape).to(device)

def sparse_batch_to_gpu_dense(batch_csr, device):
    """Build a dense GPU tensor by only transferring non-zero values over PCIe.

    toarray() (old way):  zero-fill 1.2 GB on CPU at ~50 GB/s  ≈ 24 ms
                          + transfer 1.2 GB over PCIe           ≈ 19 ms
    This function:        transfer ~1.2 MB of non-zeros         ≈ 0.02 ms
                          + torch.zeros on GPU at ~1.7 TB/s     ≈ 0.7 ms
    ~40x less time, 1000x less PCIe traffic per batch.
    """
    n_rows, n_cols = batch_csr.shape
    data    = torch.tensor(batch_csr.data,            dtype=torch.float32, device=device)
    indices = torch.tensor(batch_csr.indices,         dtype=torch.int64,   device=device)
    counts  = torch.tensor(np.diff(batch_csr.indptr), dtype=torch.int64,   device=device)
    row_ids = torch.repeat_interleave(
        torch.arange(n_rows, dtype=torch.int64, device=device), counts
    )
    dense = torch.zeros(n_rows, n_cols, dtype=torch.float32, device=device)
    dense[row_ids, indices] = data
    return dense

def knn_predict(test_mat, train_mat, train_labels, k, batch_size=2000, _train_gpu=None):
    owns_train = _train_gpu is None
    train_gpu  = scipy_sparse_to_torch(train_mat, DEVICE) if owns_train else _train_gpu

    train_labels_arr = np.array(train_labels)
    n_test           = test_mat.shape[0]
    predictions      = []

    for start in range(0, n_test, batch_size):
        end         = min(start + batch_size, n_test)
        batch_dense = sparse_batch_to_gpu_dense(test_mat[start:end], DEVICE)
        sims              = torch.sparse.mm(train_gpu, batch_dense.T).T
        top_vals, top_idx = torch.topk(sims, k, dim=1)
        for sim_scores, idx_row in zip(top_vals.cpu().numpy(), top_idx.cpu().numpy()):
            top_k_labels = train_labels_arr[idx_row]
            weights      = defaultdict(float)
            for label, w in zip(top_k_labels, sim_scores):
                weights[label] += float(w)
            predictions.append(max(weights, key=weights.get))
        if (start // batch_size) % 5 == 0:
            print(f'  {end}/{n_test} done...')

    if owns_train:
        del train_gpu
    return predictions

def best_k_search(val_mat, fit_mat, fit_labels, val_labels, k_list):
    """Sweep k values — pre-loads train matrix onto GPU once, reused across all k."""
    best_k, best_f1 = 1, 0.0
    train_gpu = scipy_sparse_to_torch(fit_mat, DEVICE)
    for k in k_list:
        preds = knn_predict(val_mat, fit_mat, fit_labels, k=k, _train_gpu=train_gpu)
        f1    = f1_score(val_labels, preds, average='macro')
        print(f'  k={k}  F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_k = f1, k
    del train_gpu
    return best_k, best_f1

print('All helper functions defined.')


All helper functions defined.


In [67]:
# ── Cell 6: BM25 Parameter Search (quick subset) ─────────────────────────────
# Find best k1 and b on 12k samples before building full matrices.

MIN_DF  = 2
BM25_K1 = 1.5
BM25_B  = 0.75
N       = len(train_tokens)
avgdl   = np.mean([len(t) for t in train_tokens])

# Shared vocab — larger for Model 1 (trigrams need more room)
vocab, idf, doc_freq = build_vocab_idf(train_tokens, min_df=MIN_DF,
                                        max_vocab=200000, mode='bm25')
print(f'Vocab size: {len(vocab)}  avgdl: {avgdl:.1f}')

random.seed(42)
q_idx = list(range(12000)); random.shuffle(q_idx)
q_fit_idx, q_val_idx = q_idx[:10000], q_idx[10000:]
q_fit_labels = [train_labels[i] for i in q_fit_idx]
q_val_labels = [train_labels[i] for i in q_val_idx]

best_params, best_qf1 = (BM25_K1, BM25_B), 0.0
for k1 in [1.2, 1.5, 2.0]:
    for b in [0.5, 0.75, 1.0]:
        qtr = build_bm25_matrix([train_tokens[i] for i in q_fit_idx], vocab, idf, k1, b, avgdl)
        qva = build_bm25_matrix([train_tokens[i] for i in q_val_idx], vocab, idf, k1, b, avgdl)
        preds = knn_predict(qva, qtr, q_fit_labels, k=11, batch_size=200)
        f1    = f1_score(q_val_labels, preds, average='macro')
        print(f'  k1={k1}  b={b}  F1={f1:.4f}')
        if f1 > best_qf1:
            best_qf1, best_params = f1, (k1, b)

BM25_K1, BM25_B = best_params
print(f'\nBest: k1={BM25_K1}  b={BM25_B}')

Vocab size: 200000  avgdl: 74.0
  200/2000 done...
  1200/2000 done...
  k1=1.2  b=0.5  F1=0.8672
  200/2000 done...
  1200/2000 done...
  k1=1.2  b=0.75  F1=0.8656
  200/2000 done...
  1200/2000 done...
  k1=1.2  b=1.0  F1=0.8652
  200/2000 done...
  1200/2000 done...
  k1=1.5  b=0.5  F1=0.8655
  200/2000 done...
  1200/2000 done...
  k1=1.5  b=0.75  F1=0.8661
  200/2000 done...
  1200/2000 done...
  k1=1.5  b=1.0  F1=0.8657
  200/2000 done...
  1200/2000 done...
  k1=2.0  b=0.5  F1=0.8661
  200/2000 done...
  1200/2000 done...
  k1=2.0  b=0.75  F1=0.8671
  200/2000 done...
  1200/2000 done...
  k1=2.0  b=1.0  F1=0.8671

Best: k1=1.2  b=0.5


In [68]:
# ── Cell 7: Build Model 1 Matrix (BM25 + uni+bi+trigrams + chi2) ──────────────
print('Building Model 1 (BM25 + trigrams)...')
# vocab_m1 uses a larger vocab than the shared vocab in Cell 6
vocab_m1, idf_m1, _ = build_vocab_idf(train_tokens, min_df=2,
                                       max_vocab=300000, mode='bm25')
avgdl_m1      = np.mean([len(t) for t in train_tokens])
train_m1_full = build_bm25_matrix(train_tokens, vocab_m1, idf_m1, BM25_K1, BM25_B, avgdl_m1)
test_m1_full  = build_bm25_matrix(test_tokens,  vocab_m1, idf_m1, BM25_K1, BM25_B, avgdl_m1)
train_m1, test_m1 = apply_chi2(train_m1_full, test_m1_full, train_labels, k=200000)
print(f'M1 shape: {train_m1.shape}')


Building Model 1 (BM25 + trigrams)...
M1 shape: (102080, 200000)


In [69]:
# ── Cell 8: Build Model 2 Matrix (sublinear TF-IDF + uni+bigrams + chi2) ──────
print('Building Model 2 (TF-IDF + bigrams)...')
train_tokens_m2 = [[t for t in toks if t.count('_') <= 1] for toks in train_tokens]
test_tokens_m2  = [[t for t in toks if t.count('_') <= 1] for toks in test_tokens]
vocab_m2, idf_m2, _ = build_vocab_idf(train_tokens_m2, min_df=MIN_DF,
                                        max_vocab=150000, mode='tfidf')
train_m2_full = build_tfidf_matrix(train_tokens_m2, vocab_m2, idf_m2)
test_m2_full  = build_tfidf_matrix(test_tokens_m2,  vocab_m2, idf_m2)
train_m2, test_m2 = apply_chi2(train_m2_full, test_m2_full, train_labels, k=120000)
print(f'M2 shape: {train_m2.shape}')

Building Model 2 (TF-IDF + bigrams)...
M2 shape: (102080, 120000)


In [70]:
# ── Cell 9: Build Model 3 Matrix (BM25 + unigrams only + chi2) ───────────────
print('Building Model 3 (BM25 + unigrams)...')
train_tokens_m3 = [[t for t in toks if '_' not in t] for toks in train_tokens]
test_tokens_m3  = [[t for t in toks if '_' not in t] for toks in test_tokens]
vocab_m3, idf_m3, _ = build_vocab_idf(train_tokens_m3, min_df=MIN_DF,
                                        max_vocab=50000, mode='bm25')
avgdl_m3      = np.mean([len(t) for t in train_tokens_m3])
train_m3_full = build_bm25_matrix(train_tokens_m3, vocab_m3, idf_m3, BM25_K1, BM25_B, avgdl_m3)
test_m3_full  = build_bm25_matrix(test_tokens_m3,  vocab_m3, idf_m3, BM25_K1, BM25_B, avgdl_m3)
train_m3, test_m3 = apply_chi2(train_m3_full, test_m3_full, train_labels,
                                k=min(50000, train_m3_full.shape[1]))
print(f'M3 shape: {train_m3.shape}')

Building Model 3 (BM25 + unigrams)...
M3 shape: (102080, 35566)


In [71]:
# ── Cell 9b: Build Model 4 Matrix (BM25 + uni+bigrams + chi2) ────────────────
# Adds ensemble diversity: BM25 scoring on uni+bigrams (no trigrams, no TF-IDF)
print('Building Model 4 (BM25 + bigrams)...')
train_tokens_m4 = [[t for t in toks if t.count('_') <= 1] for toks in train_tokens]
test_tokens_m4  = [[t for t in toks if t.count('_') <= 1] for toks in test_tokens]
vocab_m4, idf_m4, _ = build_vocab_idf(train_tokens_m4, min_df=MIN_DF,
                                        max_vocab=150000, mode='bm25')
avgdl_m4      = np.mean([len(t) for t in train_tokens_m4])
train_m4_full = build_bm25_matrix(train_tokens_m4, vocab_m4, idf_m4, BM25_K1, BM25_B, avgdl_m4)
test_m4_full  = build_bm25_matrix(test_tokens_m4,  vocab_m4, idf_m4, BM25_K1, BM25_B, avgdl_m4)
train_m4, test_m4 = apply_chi2(train_m4_full, test_m4_full, train_labels,
                                k=min(120000, train_m4_full.shape[1]))
print(f'M4 shape: {train_m4.shape}')

Building Model 4 (BM25 + bigrams)...
M4 shape: (102080, 120000)


In [72]:
# ── Cell 9c: Build Model 5 (lemmatization + BM25 + uni+bigrams + chi2) ────────
# Genuinely different from Models 1-4: uses WordNet lemmatizer instead of
# Porter stemmer, capturing actual dictionary base forms vs heuristic chopping.
print('Building Model 5 (lemmatization + BM25 + bigrams)...')

lemmatizer = WordNetLemmatizer()

def preprocess_lemma(text):
    text     = text.lower().translate(PUNCT_TABLE)
    unigrams = [lemmatizer.lemmatize(t) for t in text.split()
                if t not in STOP_WORDS and len(t) > 1]
    bigrams  = [f'{unigrams[i]}_{unigrams[i+1]}'
                for i in range(len(unigrams) - 1)]
    return unigrams + bigrams

train_tokens_m5 = [preprocess_lemma(t) for t in train_texts]
test_tokens_m5  = [preprocess_lemma(t) for t in test_texts]
avgdl_m5        = np.mean([len(t) for t in train_tokens_m5])
vocab_m5, idf_m5, _ = build_vocab_idf(train_tokens_m5, min_df=MIN_DF,
                                        max_vocab=150000, mode='bm25')
train_m5_full = build_bm25_matrix(train_tokens_m5, vocab_m5, idf_m5, BM25_K1, BM25_B, avgdl_m5)
test_m5_full  = build_bm25_matrix(test_tokens_m5,  vocab_m5, idf_m5, BM25_K1, BM25_B, avgdl_m5)
train_m5, test_m5 = apply_chi2(train_m5_full, test_m5_full, train_labels,
                                k=min(120000, train_m5_full.shape[1]))
print(f'M5 shape: {train_m5.shape}')

Building Model 5 (lemmatization + BM25 + bigrams)...
M5 shape: (102080, 120000)


In [73]:
# ── Cell 9d: Build Model 6 (raw words, no stemming + BM25 + uni+bigrams) ──────
# No stemming or lemmatizing — keeps full word forms like "running", "better",
# "stocks", "markets". Completely different vocabulary from M1-M5.
print('Building Model 6 (raw words + BM25 + bigrams)...')

def preprocess_raw(text):
    text     = text.lower().translate(PUNCT_TABLE)
    unigrams = [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]
    bigrams  = [f'{unigrams[i]}_{unigrams[i+1]}'
                for i in range(len(unigrams) - 1)]
    return unigrams + bigrams

train_tokens_m6 = [preprocess_raw(t) for t in train_texts]
test_tokens_m6  = [preprocess_raw(t) for t in test_texts]
avgdl_m6        = np.mean([len(t) for t in train_tokens_m6])
vocab_m6, idf_m6, _ = build_vocab_idf(train_tokens_m6, min_df=MIN_DF,
                                        max_vocab=150000, mode='bm25')
train_m6_full = build_bm25_matrix(train_tokens_m6, vocab_m6, idf_m6, BM25_K1, BM25_B, avgdl_m6)
test_m6_full  = build_bm25_matrix(test_tokens_m6,  vocab_m6, idf_m6, BM25_K1, BM25_B, avgdl_m6)
train_m6, test_m6 = apply_chi2(train_m6_full, test_m6_full, train_labels,
                                k=min(120000, train_m6_full.shape[1]))
print(f'M6 shape: {train_m6.shape}')

Building Model 6 (raw words + BM25 + bigrams)...
M6 shape: (102080, 120000)


In [74]:
# ── Cell 9e: Build Model 7 (lemmatization + TF-IDF + uni+bigrams) ─────────────
# TF-IDF scoring with lemmatized tokens — combines M5's preprocessing
# with M2's scoring function for another distinct representation.
print('Building Model 7 (lemmatization + TF-IDF + bigrams)...')

vocab_m7, idf_m7, _ = build_vocab_idf(train_tokens_m5, min_df=MIN_DF,
                                        max_vocab=150000, mode='tfidf')
train_m7_full = build_tfidf_matrix(train_tokens_m5, vocab_m7, idf_m7)
test_m7_full  = build_tfidf_matrix(test_tokens_m5,  vocab_m7, idf_m7)
train_m7, test_m7 = apply_chi2(train_m7_full, test_m7_full, train_labels,
                                k=min(120000, train_m7_full.shape[1]))
print(f'M7 shape: {train_m7.shape}')

Building Model 7 (lemmatization + TF-IDF + bigrams)...
M7 shape: (102080, 120000)


In [75]:
# ── Cell 9f: Build Model 8 (TF-IDF + uni+bi+trigrams + chi2) ──────────────────
# M1 uses BM25 + trigrams; M8 uses TF-IDF + trigrams — same features, different
# scoring. Adds diversity without needing new preprocessing.
print('Building Model 8 (TF-IDF + trigrams)...')
vocab_m8, idf_m8, _ = build_vocab_idf(train_tokens, min_df=2,
                                       max_vocab=300000, mode='tfidf')
train_m8_full = build_tfidf_matrix(train_tokens, vocab_m8, idf_m8)
test_m8_full  = build_tfidf_matrix(test_tokens,  vocab_m8, idf_m8)
train_m8, test_m8 = apply_chi2(train_m8_full, test_m8_full, train_labels, k=200000)
print(f'M8 shape: {train_m8.shape}')


Building Model 8 (TF-IDF + trigrams)...
M8 shape: (102080, 200000)


In [76]:
# ── Cell 9g: Build Model 9 (LSI/LSA — SVD-reduced TF-IDF + cosine k-NN) ───────
# Reduces TF-IDF + trigrams to a dense 300-dim space via truncated SVD.
# Captures latent semantic structure: synonyms and topically-related terms
# end up close even when they never co-occur. Different signal than M1-M8.
from sklearn.decomposition import TruncatedSVD

print('Building Model 9 (LSI: SVD-300 on TF-IDF + trigrams)...')

svd = TruncatedSVD(n_components=300, random_state=42,
                   algorithm='randomized', n_iter=5)
train_m9_raw = svd.fit_transform(train_m8_full)
test_m9_raw  = svd.transform(test_m8_full)
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.4f}')

def l2_normalize_rows(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms

train_m9 = l2_normalize_rows(train_m9_raw).astype(np.float32)
test_m9  = l2_normalize_rows(test_m9_raw).astype(np.float32)
print(f'M9 shape: {train_m9.shape}')

def knn_predict_dense(test_dense, train_dense, train_labels, k,
                      batch_size=2000, _train_gpu=None):
    owns_train = _train_gpu is None
    train_gpu  = (torch.tensor(train_dense, dtype=torch.float32, device=DEVICE)
                  if owns_train else _train_gpu)
    train_labels_arr = np.array(train_labels)
    n_test           = test_dense.shape[0]
    predictions      = []

    for start in range(0, n_test, batch_size):
        end   = min(start + batch_size, n_test)
        batch = torch.tensor(test_dense[start:end], dtype=torch.float32, device=DEVICE)
        sims              = batch @ train_gpu.T
        top_vals, top_idx = torch.topk(sims, k, dim=1)
        for sim_scores, idx_row in zip(top_vals.cpu().numpy(), top_idx.cpu().numpy()):
            top_k_labels = train_labels_arr[idx_row]
            weights      = defaultdict(float)
            for label, w in zip(top_k_labels, sim_scores):
                weights[label] += float(w)
            predictions.append(max(weights, key=weights.get))
        if (start // batch_size) % 5 == 0:
            print(f'  {end}/{n_test} done...')

    if owns_train:
        del train_gpu
    return predictions

def best_k_search_dense(val_mat, fit_mat, fit_labels, val_labels, k_list):
    best_k, best_f1 = 1, 0.0
    train_gpu = torch.tensor(fit_mat, dtype=torch.float32, device=DEVICE)
    for k in k_list:
        preds = knn_predict_dense(val_mat, fit_mat, fit_labels, k=k, _train_gpu=train_gpu)
        f1    = f1_score(val_labels, preds, average='macro')
        print(f'  k={k}  F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_k = f1, k
    del train_gpu
    return best_k, best_f1


Building Model 9 (LSI: SVD-300 on TF-IDF + trigrams)...
SVD explained variance: 0.1043
M9 shape: (102080, 300)


In [77]:
# ── Cell 9h: Build Model 10 (LSI on BM25 + trigrams) ──────────────────────────
# Second LSI view — SVD over BM25-scored features instead of TF-IDF. BM25's
# saturation function emphasizes different term-document structure than raw
# TF-IDF, so the latent topics SVD extracts are different too.
print('Building Model 10 (LSI: SVD-300 on BM25 + trigrams)...')

svd_m10 = TruncatedSVD(n_components=300, random_state=17,
                        algorithm='randomized', n_iter=5)
train_m10_raw = svd_m10.fit_transform(train_m1_full)
test_m10_raw  = svd_m10.transform(test_m1_full)
print(f'SVD explained variance: {svd_m10.explained_variance_ratio_.sum():.4f}')

train_m10 = l2_normalize_rows(train_m10_raw).astype(np.float32)
test_m10  = l2_normalize_rows(test_m10_raw).astype(np.float32)
print(f'M10 shape: {train_m10.shape}')


Building Model 10 (LSI: SVD-300 on BM25 + trigrams)...
SVD explained variance: 0.0829
M10 shape: (102080, 300)


In [78]:
# ── Cell 9i: Score-level ensemble helpers ─────────────────────────────────────
# Hard-vote ensembles throw away per-model confidence. These functions return
# per-class similarity sums (shape: n_test × n_classes) so the ensemble can
# average soft probabilities instead — confident models get proportionally
# more say than unsure ones.

CLASS_LABELS = np.array([1, 2, 3, 4])

def knn_predict_scores(test_mat, train_mat, train_labels, k,
                       batch_size=2000, _train_gpu=None):
    owns_train = _train_gpu is None
    train_gpu  = scipy_sparse_to_torch(train_mat, DEVICE) if owns_train else _train_gpu
    train_labels_arr = np.array(train_labels)
    n_test           = test_mat.shape[0]
    scores           = np.zeros((n_test, len(CLASS_LABELS)), dtype=np.float32)

    for start in range(0, n_test, batch_size):
        end         = min(start + batch_size, n_test)
        batch_dense = sparse_batch_to_gpu_dense(test_mat[start:end], DEVICE)
        sims              = torch.sparse.mm(train_gpu, batch_dense.T).T
        top_vals, top_idx = torch.topk(sims, k, dim=1)
        top_vals_np = top_vals.cpu().numpy()
        top_lbls_np = train_labels_arr[top_idx.cpu().numpy()]
        for c_idx, c in enumerate(CLASS_LABELS):
            scores[start:end, c_idx] = (top_vals_np * (top_lbls_np == c)).sum(axis=1)

    if owns_train:
        del train_gpu
    return scores

def knn_predict_dense_scores(test_dense, train_dense, train_labels, k,
                             batch_size=2000, _train_gpu=None):
    owns_train = _train_gpu is None
    train_gpu  = (torch.tensor(train_dense, dtype=torch.float32, device=DEVICE)
                  if owns_train else _train_gpu)
    train_labels_arr = np.array(train_labels)
    n_test           = test_dense.shape[0]
    scores           = np.zeros((n_test, len(CLASS_LABELS)), dtype=np.float32)

    for start in range(0, n_test, batch_size):
        end   = min(start + batch_size, n_test)
        batch = torch.tensor(test_dense[start:end], dtype=torch.float32, device=DEVICE)
        sims              = batch @ train_gpu.T
        top_vals, top_idx = torch.topk(sims, k, dim=1)
        top_vals_np = top_vals.cpu().numpy()
        top_lbls_np = train_labels_arr[top_idx.cpu().numpy()]
        for c_idx, c in enumerate(CLASS_LABELS):
            scores[start:end, c_idx] = (top_vals_np * (top_lbls_np == c)).sum(axis=1)

    if owns_train:
        del train_gpu
    return scores

def l1_normalize_rows(arr):
    """Normalize each row to sum to 1 (sims -> probabilities)."""
    sums = arr.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1
    return arr / sums

print('Score-level helpers defined.')


Score-level helpers defined.


In [79]:
# ── Cell 9j: Pseudo-Relevance Feedback (PRF) ──────────────────────────────────
# Rocchio-style query expansion: first k-NN pass finds top-N neighbors, then
# each query is replaced by alpha*query + beta*centroid(neighbors) and re-
# queried. Pulls noisy/short queries toward their topical cluster.

PRF_N     = 10     # neighbors used to build centroid
PRF_ALPHA = 0.9    # weight on original query
PRF_BETA  = 0.1    # weight on centroid

def build_prf_queries(query_mat, corpus_mat, n_prf=PRF_N,
                      alpha=PRF_ALPHA, beta=PRF_BETA, batch_size=2000):
    """Return a new query matrix where each row is an expanded query."""
    corpus_gpu = scipy_sparse_to_torch(corpus_mat, DEVICE)
    n_query    = query_mat.shape[0]
    n_corpus   = corpus_mat.shape[0]
    all_rows, all_cols, all_vals = [], [], []

    for start in range(0, n_query, batch_size):
        end         = min(start + batch_size, n_query)
        batch_dense = sparse_batch_to_gpu_dense(query_mat[start:end], DEVICE)
        sims              = torch.sparse.mm(corpus_gpu, batch_dense.T).T
        top_vals, top_idx = torch.topk(sims, n_prf, dim=1)
        top_vals_np = top_vals.cpu().numpy()
        top_idx_np  = top_idx.cpu().numpy()

        # similarity-weighted centroid via sparse W @ corpus
        row_weights = top_vals_np / top_vals_np.sum(axis=1, keepdims=True).clip(min=1e-9)
        batch_rows  = np.repeat(np.arange(end - start), n_prf)
        W_batch     = csr_matrix(
            (row_weights.flatten(), (batch_rows, top_idx_np.flatten())),
            shape=(end - start, n_corpus), dtype=np.float32)
        centroids   = W_batch @ corpus_mat
        expanded    = alpha * query_mat[start:end] + beta * centroids

        coo = expanded.tocoo()
        all_rows.append(coo.row + start)
        all_cols.append(coo.col)
        all_vals.append(coo.data)
        if (start // batch_size) % 5 == 0:
            print(f'  PRF expand {end}/{n_query}...')

    rows = np.concatenate(all_rows)
    cols = np.concatenate(all_cols)
    vals = np.concatenate(all_vals)
    expanded = csr_matrix((vals, (rows, cols)),
                           shape=query_mat.shape, dtype=np.float32)
    del corpus_gpu
    return renormalize(expanded)

print(f'PRF defined. alpha={PRF_ALPHA}  beta={PRF_BETA}  N={PRF_N}')


PRF defined. alpha=0.9  beta=0.1  N=10


In [80]:
# ── Cell 10: Validate Models + Ensemble ───────────────────────────────────────
random.seed(42)
indices = list(range(len(train_labels))); random.shuffle(indices)
split   = int(0.8 * len(indices))
fit_idx, val_idx = indices[:split], indices[split:]
fit_labels = [train_labels[i] for i in fit_idx]
val_labels = [train_labels[i] for i in val_idx]

K_LIST       = [11] + list(range(21, 42))
K_LIST_DENSE = [11, 21, 31, 41, 51, 71, 101]

print('--- Model 1 ---')
best_k_m1, best_f1_m1 = best_k_search(
    train_m1[val_idx], train_m1[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M1 best: k={best_k_m1}  F1={best_f1_m1:.4f}\n')

print('--- Model 2 ---')
best_k_m2, best_f1_m2 = best_k_search(
    train_m2[val_idx], train_m2[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M2 best: k={best_k_m2}  F1={best_f1_m2:.4f}\n')

print('--- Model 3 ---')
best_k_m3, best_f1_m3 = best_k_search(
    train_m3[val_idx], train_m3[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M3 best: k={best_k_m3}  F1={best_f1_m3:.4f}\n')

print('--- Model 4 ---')
best_k_m4, best_f1_m4 = best_k_search(
    train_m4[val_idx], train_m4[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M4 best: k={best_k_m4}  F1={best_f1_m4:.4f}\n')

print('--- Model 5 ---')
best_k_m5, best_f1_m5 = best_k_search(
    train_m5[val_idx], train_m5[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M5 best: k={best_k_m5}  F1={best_f1_m5:.4f}\n')

print('--- Model 6 ---')
best_k_m6, best_f1_m6 = best_k_search(
    train_m6[val_idx], train_m6[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M6 best: k={best_k_m6}  F1={best_f1_m6:.4f}\n')

print('--- Model 7 ---')
best_k_m7, best_f1_m7 = best_k_search(
    train_m7[val_idx], train_m7[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M7 best: k={best_k_m7}  F1={best_f1_m7:.4f}\n')

print('--- Model 8 ---')
best_k_m8, best_f1_m8 = best_k_search(
    train_m8[val_idx], train_m8[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M8 best: k={best_k_m8}  F1={best_f1_m8:.4f}\n')

print('--- Model 9 (LSI TF-IDF) ---')
best_k_m9, best_f1_m9 = best_k_search_dense(
    train_m9[val_idx], train_m9[fit_idx], fit_labels, val_labels, K_LIST_DENSE)
print(f'M9 best: k={best_k_m9}  F1={best_f1_m9:.4f}\n')

print('--- Model 10 (LSI BM25) ---')
best_k_m10, best_f1_m10 = best_k_search_dense(
    train_m10[val_idx], train_m10[fit_idx], fit_labels, val_labels, K_LIST_DENSE)
print(f'M10 best: k={best_k_m10}  F1={best_f1_m10:.4f}\n')

print('--- Model 11 (PRF on M1) ---')
# Expand validation queries using their top-N neighbors in the fit corpus
val_m1_prf = build_prf_queries(train_m1[val_idx], train_m1[fit_idx])
best_k_m11, best_f1_m11 = best_k_search(
    val_m1_prf, train_m1[fit_idx], fit_labels, val_labels, K_LIST)
print(f'M11 best: k={best_k_m11}  F1={best_f1_m11:.4f}\n')

# Ensemble — score-level combination (soft probabilities, not hard votes)
print('--- Ensemble (score-level) ---')
sc1  = knn_predict_scores(train_m1[val_idx], train_m1[fit_idx], fit_labels, k=best_k_m1)
sc2  = knn_predict_scores(train_m2[val_idx], train_m2[fit_idx], fit_labels, k=best_k_m2)
sc3  = knn_predict_scores(train_m3[val_idx], train_m3[fit_idx], fit_labels, k=best_k_m3)
sc4  = knn_predict_scores(train_m4[val_idx], train_m4[fit_idx], fit_labels, k=best_k_m4)
sc5  = knn_predict_scores(train_m5[val_idx], train_m5[fit_idx], fit_labels, k=best_k_m5)
sc6  = knn_predict_scores(train_m6[val_idx], train_m6[fit_idx], fit_labels, k=best_k_m6)
sc7  = knn_predict_scores(train_m7[val_idx], train_m7[fit_idx], fit_labels, k=best_k_m7)
sc8  = knn_predict_scores(train_m8[val_idx], train_m8[fit_idx], fit_labels, k=best_k_m8)
sc9  = knn_predict_dense_scores(train_m9[val_idx], train_m9[fit_idx], fit_labels, k=best_k_m9)
sc10 = knn_predict_dense_scores(train_m10[val_idx], train_m10[fit_idx], fit_labels, k=best_k_m10)
sc11 = knn_predict_scores(val_m1_prf, train_m1[fit_idx], fit_labels, k=best_k_m11)

w1, w2, w3 = best_f1_m1, best_f1_m2, best_f1_m3
w4, w5, w6, w7, w8 = best_f1_m4, best_f1_m5, best_f1_m6, best_f1_m7, best_f1_m8
w9, w10, w11 = best_f1_m9, best_f1_m10, best_f1_m11

probs_val = [l1_normalize_rows(s) for s in
             [sc1, sc2, sc3, sc4, sc5, sc6, sc7, sc8, sc9, sc10, sc11]]
weights   = [w1, w2, w3, w4, w5, w6, w7, w8, w9, w10, w11]

def ensemble_with_temperature(probs_list, weights, T):
    combined = np.zeros_like(probs_list[0])
    for p, w in zip(probs_list, weights):
        combined += w * l1_normalize_rows(p ** T)
    return combined

print('\nTemperature sweep:')
best_T, best_ens_f1 = 1.0, 0.0
for T in [1.0, 1.5, 2.0, 3.0, 5.0]:
    combined_val = ensemble_with_temperature(probs_val, weights, T)
    ens_val = CLASS_LABELS[combined_val.argmax(axis=1)]
    f1 = f1_score(val_labels, ens_val, average='macro')
    print(f'  T={T:4.1f}  F1={f1:.4f}')
    if f1 > best_ens_f1:
        best_ens_f1, best_T = f1, T

print(f'\nBest: T={best_T}  Ensemble F1 = {best_ens_f1:.4f}')
print(f'  M1={best_f1_m1:.4f}  M2={best_f1_m2:.4f}  M3={best_f1_m3:.4f}  M4={best_f1_m4:.4f}')
print(f'  M5={best_f1_m5:.4f}  M6={best_f1_m6:.4f}  M7={best_f1_m7:.4f}  M8={best_f1_m8:.4f}')
print(f'  M9={best_f1_m9:.4f}  M10={best_f1_m10:.4f}  M11={best_f1_m11:.4f}')


--- Model 1 ---
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=11  F1=0.9216
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=21  F1=0.9209
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=22  F1=0.9210
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=23  F1=0.9206
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=24  F1=0.9212
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=25  F1=0.9207
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=26  F1=0.9206
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=27  F1=0.9201
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=28  F1=0.9202
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=29  F1=0.9195
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=30  F1=0.9195
  2000/20416 done...
  12000/20416 done...
  20416/20416 done...
  k=31

In [81]:
# ── Cell 11: Final Predictions (11-model score-level ensemble + temperature) ──
print('M1 on full test...')
sc_t1  = knn_predict_scores(test_m1, train_m1, train_labels, k=best_k_m1)
print('M2 on full test...')
sc_t2  = knn_predict_scores(test_m2, train_m2, train_labels, k=best_k_m2)
print('M3 on full test...')
sc_t3  = knn_predict_scores(test_m3, train_m3, train_labels, k=best_k_m3)
print('M4 on full test...')
sc_t4  = knn_predict_scores(test_m4, train_m4, train_labels, k=best_k_m4)
print('M5 on full test...')
sc_t5  = knn_predict_scores(test_m5, train_m5, train_labels, k=best_k_m5)
print('M6 on full test...')
sc_t6  = knn_predict_scores(test_m6, train_m6, train_labels, k=best_k_m6)
print('M7 on full test...')
sc_t7  = knn_predict_scores(test_m7, train_m7, train_labels, k=best_k_m7)
print('M8 on full test...')
sc_t8  = knn_predict_scores(test_m8, train_m8, train_labels, k=best_k_m8)
print('M9 on full test...')
sc_t9  = knn_predict_dense_scores(test_m9, train_m9, train_labels, k=best_k_m9)
print('M10 on full test...')
sc_t10 = knn_predict_dense_scores(test_m10, train_m10, train_labels, k=best_k_m10)
print('M11 on full test (PRF)...')
test_m1_prf = build_prf_queries(test_m1, train_m1)
sc_t11 = knn_predict_scores(test_m1_prf, train_m1, train_labels, k=best_k_m11)

probs_test = [l1_normalize_rows(s) for s in
              [sc_t1, sc_t2, sc_t3, sc_t4, sc_t5, sc_t6,
               sc_t7, sc_t8, sc_t9, sc_t10, sc_t11]]

combined_test    = ensemble_with_temperature(probs_test, weights, best_T)
test_predictions = [int(x) for x in CLASS_LABELS[combined_test.argmax(axis=1)]]

print(f'Total: {len(test_predictions)}  Distribution: {Counter(test_predictions)}  T={best_T}')


M1 on full test...
M2 on full test...
M3 on full test...
M4 on full test...
M5 on full test...
M6 on full test...
M7 on full test...
M8 on full test...
M9 on full test...
M10 on full test...
M11 on full test (PRF)...
  PRF expand 2000/25520...
  PRF expand 12000/25520...
  PRF expand 22000/25520...
Total: 25520  Distribution: Counter({2: 6584, 4: 6455, 3: 6296, 1: 6185})  T=1.5


In [82]:
# ── Cell 12: Save Output ──────────────────────────────────────────────────────
with open('predictions.dat', 'w') as f:
    for pred in test_predictions:
        f.write(f'{pred}\n')

with open('format.dat') as f:
    n_fmt = sum(1 for _ in f)

print(f'Saved predictions.dat — {len(test_predictions)} lines')
print('Match!' if len(test_predictions) == n_fmt else 'WARNING: line count mismatch!')

Saved predictions.dat — 25520 lines
Match!
